# Is December really the worst month to fly?

Everyone knows winter is the worst time to fly. This notebook checks it with **every flight at
UK airports from January 2006 to September 2026**: 33 million take-offs and landings, including
every flight between the UK and the rest of Europe, from the UK Civil Aviation Authority's
punctuality statistics. One table per month, one row per airport × route × airline ×
arrival/departure, with the share of flights in each delay band.

**The yardstick**

* **Late** = more than 15 minutes after the timetable at the gate (the industry standard, and
  the CAA's own "early to 15 minutes late" is on time)
* **3 hours+** = more than 180 minutes, where EU/UK261 compensation can start
* **Cancelled** = called off less than 24 hours before departure. Only counted from 2018 on;
  before that, cancelled flights just aren't in the files
* **A flight** = one take-off or one landing at a UK airport. A domestic flight between two
  reporting airports counts at both ends, the same as in the CAA's own tables
* Long series (month × year, every December) only use the **8 airports in every month**,
  because the CAA went from 10 to 25 reporting airports in 2014

The data is monthly: there's no Christmas week or Christmas Eve, only December as a whole.

In [ ]:
import glob
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

DATA = Path("data") / "caa"
LAST12 = (202510, 202609)          # the latest 12 months published
RECENT = (202301, 202609)          # after the post-Covid recovery
COVID = (202003, 202112)           # left out of every average
MIN_FLIGHTS = 3000                 # per airline / destination in LAST12 before it gets ranked
MONTHS = "Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec".split()

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE = "#4F79B8", "#E8641D"
pd.set_option("display.float_format", "{:.3f}".format)

def hbar(series, title, fmt="{:.0%}", hot=None, xmax=None, labels=None):
    """Horizontal bars, biggest first at the top. hot = labels to colour orange."""
    hot = [hot] if isinstance(hot, str) else (hot or [])
    names, values = list(series.index)[::-1], [float(v) for v in series.values][::-1]
    xmax = xmax or max(values) * 1.18
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(values) + 1.2))
    ax.barh([str(n) for n in names], values, color=[ORANGE if n in hot else BLUE for n in names])
    ax.set(title=title, xlim=(0, xmax))
    if "%" in fmt:
        ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}" if xmax > 0.1 else f"{v:.1%}")
    texts = (labels[::-1] if labels is not None else [fmt.format(v) for v in values])
    for i, (v, t) in enumerate(zip(values, texts)):
        ax.text(v + xmax * 0.01, i, t, va="center")
    plt.show()

## 1. Download

One "Full Analysis Arr Dep" CSV per month from the
[CAA's punctuality statistics](https://www.caa.co.uk/data-and-analysis/uk-aviation-market/flight-punctuality/),
about 1 MB each, ~250 files. The CAA has no fixed file names, so this reads each year's page and
picks the monthly links. Months already downloaded are skipped, so you can re-run it to fill gaps.

The CAA asks to be credited ("Source: UK Civil Aviation Authority") and says the statistics may
not be sold on. Please don't re-upload the raw files: download them from the CAA.

In [ ]:
SITE = "https://www.caa.co.uk"
PAGE = SITE + "/data-and-analysis/uk-aviation-market/flight-punctuality/uk-flight-punctuality-statistics/{}/"
MONTHLY = re.compile(r"^(\d{6})\s*(?:Punctuality Statistics\s*)?Full\s*Analysis\s*Arr(?:ival)?\s*(?:/\s*)?Dep", re.I)
HEADERS = {"User-Agent": "Mozilla/5.0 (research notebook)"}


def month_links(year):
    html = requests.get(PAGE.format(year), headers=HEADERS, timeout=60).text
    found = {}
    for href, body in re.findall(r'href="(/Documents/Download/[^"]+)"(.*?)</a>', html, re.S):
        label = " ".join(re.sub(r"<[^>]+>", " ", body).replace(">", " ").split())
        m = MONTHLY.match(label)
        if m and "CSV" in label.upper():
            found.setdefault(m.group(1), SITE + href)
    return found


DATA.mkdir(parents=True, exist_ok=True)
for year in range(2006, 2027):
    try:
        links = month_links(year)
    except Exception as e:
        print(f"{year}: year page failed ({e})")
        continue
    new = 0
    for ym, url in sorted(links.items()):
        path = DATA / f"{ym}.csv"
        if path.exists() and path.stat().st_size > 1000:
            continue
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
            r.raise_for_status()
            path.write_bytes(r.content)
            new += 1
            time.sleep(0.5)
        except Exception as e:
            print(f"  {ym} failed ({e}): run this cell again")
    print(f"{year}: {len(links)} monthly files, {new} new")

## 2. Prepare: one table, one yardstick

Two things changed in the source and are handled here:

* **2018**: delay is measured gate-to-gate (before: runway times minus a standard taxi time),
  there are more delay bands and cancellations were added. The bands are merged back to the old
  ones for long series. Keep the break in mind on any chart that crosses 2018.
* **Encodings**: some older files are Windows-1252, not UTF-8.

Airlines are grouped into brands (easyJet UK / Europe / Switzerland are one easyJet), and
"Teesside International" and "Durham Tees Valley" are one airport.

In [ ]:
EUROPE = {
    "ALBANIA", "AUSTRIA", "BELARUS", "BELGIUM", "BOSNIA-HERZEGOVINA", "BULGARIA", "CROATIA",
    "CYPRUS", "CZECH REPUBLIC", "DENMARK", "ESTONIA", "FAROE ISLANDS", "FED REP YUGO SERBIA M'ENEGRO",
    "FINLAND", "FRANCE", "GERMANY", "GIBRALTAR", "GREECE", "HUNGARY", "ICELAND", "IRISH REPUBLIC",
    "ITALY", "KOSOVO", "LATVIA", "LITHUANIA", "LUXEMBOURG", "MACEDONIA", "MALTA", "NETHERLANDS",
    "NORWAY", "POLAND", "PORTUGAL(EXCLUDING MADEIRA)", "PORTUGAL(MADEIRA)", "REPUBLIC OF MOLDOVA",
    "REPUBLIC OF MONTENEGRO", "REPUBLIC OF SERBIA", "ROMANIA", "RUMANIA", "SLOVAK REPUBLIC",
    "SLOVENIA", "SPAIN", "SPAIN(CANARY ISLANDS)", "SWEDEN", "SWITZERLAND", "TURKEY", "UKRAINE",
}
BRANDS = [
    ("easyJet", r"^EASYJET"), ("Ryanair", r"^RYANAIR|^BUZZ|^MALTA AIR"),
    ("British Airways", r"^BRITISH AIRWAYS|^BA CITYFLYER|^BA EUROFLYER"),
    ("Jet2", r"^JET2"), ("TUI", r"^TUI|^THOMSON"), ("Wizz Air", r"^WIZZ"),
    ("Aer Lingus", r"^AER LINGUS"), ("KLM", r"^KLM"), ("Lufthansa", r"^LUFTHANSA"),
    ("Loganair", r"^LOGANAIR"), ("Vueling", r"^VUELING"), ("Norwegian", r"^NORWEGIAN"),
    ("Air France", r"^AIR FRANCE"), ("Eurowings", r"^EUROWINGS"), ("SAS", r"^SAS\b"),
    ("Swiss", r"^SWISS"), ("Iberia", r"^IBERIA"), ("Virgin Atlantic", r"^VIRGIN ATLANTIC"),
    ("Emirates", r"^EMIRATES"), ("Aurigny", r"^AURIGNY"), ("Eastern Airways", r"^EASTERN AIRWAYS"),
    ("Blue Islands", r"^BLUE ISLANDS"), ("Flybe", r"^FLYBE"), ("Thomas Cook", r"^THOMAS COOK"),
    ("Monarch", r"^MONARCH"), ("TAP", r"^TAP"), ("Aegean", r"^AEGEAN"), ("ITA", r"^ITA\b|^ALITALIA"),
    ("Turkish Airlines", r"^TURKISH"), ("Pegasus", r"^PEGASUS"), ("Brussels Airlines", r"^BRUSSELS"),
    ("Finnair", r"^FINNAIR"), ("Icelandair", r"^ICELANDAIR"), ("airBaltic", r"^AIR BALTIC"),
    ("LOT", r"^LOT\b|^POLSKIE"), ("Austrian", r"^AUSTRIAN"), ("Volotea", r"^VOLOTEA"),
]
NAME_FIX = {"TEESSIDE INTERNATIONAL AIRPORT": "DURHAM TEES VALLEY"}

# Delay bands, merged so the pre-2018 and 2018+ files line up (share of matched flights, %)
NEW_BANDS = {
    "ontime": ["flights_more_than_15_minutes_early_percent",
               "flights_15_minutes_early_to_1_minute_early_percent",
               "flights_0_to_15_minutes_late_percent"],
    "b16_30": ["flights_between_16_and_30_minutes_late_percent"],
    "b31_60": ["flights_between_31_and_60_minutes_late_percent"],
    "b61_120": ["flights_between_61_and_120_minutes_late_percent"],
    "b121_180": ["flights_between_121_and_180_minutes_late_percent"],
    "b181_360": ["flights_between_181_and_360_minutes_late_percent"],
    "b360": ["flights_more_than_360_minutes_late_percent"],
}
OLD_BANDS = {
    "ontime": ["early_to_15_mins_late_percent"], "b16_30": ["flts_16_to_30_mins_late_percent"],
    "b31_60": ["flts_31_to_60_mins_late_percent"], "b61_180": ["flts_61_to_180_mins_late_percent"],
    "b181_360": ["flts_181_to_360_mins_late_percent"], "b360": ["more_than_360_mins_late_percent"],
}
BANDS = ["ontime", "b16_30", "b31_60", "b61_120", "b121_180", "b61_180", "b181_360", "b360"]
OVER = {15: ["b16_30", "b31_60", "b61_180", "b181_360", "b360"],
        30: ["b31_60", "b61_180", "b181_360", "b360"], 60: ["b61_180", "b181_360", "b360"],
        180: ["b181_360", "b360"], 360: ["b360"]}


def brand(name):
    for b, pat in BRANDS:
        if re.search(pat, name):
            return b
    return name.title()


def read_month(path):
    try:
        d = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        d = pd.read_csv(path, encoding="cp1252", low_memory=False)
    d.columns = [c.strip().lower() for c in d.columns]
    new = "number_flights_cancelled" in d.columns
    n = pd.to_numeric(d["number_flights_matched"], errors="coerce").fillna(0)
    out = pd.DataFrame({
        "ym": int(os.path.basename(path)[:6]),
        "airport": d["reporting_airport"].str.strip().replace(NAME_FIX),
        "country": d["origin_destination_country"].fillna("Unknown").str.strip(),
        "dest": d["origin_destination"].fillna("Unknown").str.strip(),
        "airline": d["airline_name"].fillna("").str.strip(),
        "ad": d["arrival_departure"].str.strip(),
        "sc": d["scheduled_charter"].str.strip(),
        "n": n,
        "cancelled": pd.to_numeric(d["number_flights_cancelled"], errors="coerce").fillna(0)
        if new else np.nan,
    })
    for band, cols in (NEW_BANDS if new else OLD_BANDS).items():
        out[band] = sum(pd.to_numeric(d[c], errors="coerce").fillna(0) for c in cols) / 100 * n
    if new:
        out["b61_180"] = out["b61_120"] + out["b121_180"]
    else:
        out["b61_120"] = out["b121_180"] = np.nan
    return out[out["n"] > 0]


files = sorted(f for f in glob.glob(str(DATA / ("[0-9]" * 6 + ".csv"))) if os.path.getsize(f) > 1000)
d = pd.concat([read_month(f) for f in files], ignore_index=True)
d["year"], d["month"] = d["ym"] // 100, d["ym"] % 100
d["region"] = np.where(d["country"] == "UNITED KINGDOM", "UK",
                       np.where(d["country"].isin(EUROPE), "Europe", "Long-haul"))
d["brand"] = d["airline"].map({a: brand(a) for a in d["airline"].unique()})
d["covid"] = d["ym"].between(*COVID)
core = sorted(set.intersection(*[set(a) for a in d.groupby("ym")["airport"].unique()]))
d["core"] = d["airport"].isin(core)
print(f"{len(files)} months, {d.n.sum():,.0f} flights, {d.dest.nunique():,} destinations")
print("Airports in every month:", ", ".join(a.title() for a in core))


def summarise(g):
    """Flight-weighted shares for a group of rows."""
    s = g[["n", "cancelled"] + BANDS].sum(min_count=1)
    n = s["n"]
    return pd.Series({
        "flights": int(n), "late15": 1 - s["ontime"] / n,
        "late60": sum(s[b] for b in OVER[60]) / n, "late180": sum(s[b] for b in OVER[180]) / n,
        "cancelled": s["cancelled"] / (n + s["cancelled"]) if g["cancelled"].notna().all() else np.nan})


def by(df, keys):
    return df.groupby(keys).apply(summarise, include_groups=False)


l12 = d[d.ym.between(*LAST12)]
rec = d[d.ym.between(*RECENT)]
decs = d[(d.month == 12) & d.year.between(2023, 2025)]       # the last three Decembers

## Finding 1: December used to be the worst

Every month since 2006 at the 8 airports in every month. The lighter, the more flights late.

In [ ]:
grid = by(d[d.core], ["year", "month"])
heat = grid.late15.unstack()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heat.values, cmap="Blues_r", aspect="auto")
ax.set(xticks=range(12), xticklabels=MONTHS, yticks=range(len(heat)), yticklabels=heat.index,
       title="Share of flights more than 15 minutes late (8 airports in every month)")
ax.grid(False)
ax.add_patch(plt.Rectangle((10.5, -0.5), 1, len(heat), fill=False, ec=ORANGE, lw=3))
plt.colorbar(im, format=lambda v, _: f"{v:.0%}")
plt.show()

nc = d[d.core & ~d.covid]
dec = by(nc[nc.month == 12], ["year"])
dec["rank_in_year"] = [int(grid.loc[y].late15.rank(ascending=False)[12]) for y in dec.index]
print("Years where December was the worst month:", list(dec.index[dec.rank_in_year == 1]))
dec

## Finding 2: now it's summer

The same 8 airports, 2023 to September 2026 vs 2010–2019.

In [ ]:
now = by(rec[rec.core], ["month"])
then = by(d[d.core & d.year.between(2010, 2019)], ["month"])
t = pd.DataFrame({"2010–2019": then.late15, "2023–2026": now.late15}).set_axis(MONTHS)
ax = t.plot.bar(rot=0, color=["#B8C7E0", BLUE], title="Flights more than 15 minutes late, by month")
ax.patches[11 + 12].set_color(ORANGE)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.show()
print("Since 2023, worst to best:", " · ".join(f"{MONTHS[m - 1]} {v:.0%}"
                                              for m, v in now.late15.sort_values(ascending=False).items()))

## Finding 3: fewer delays, not fewer bad ones

The chance a flight is more than X minutes late, July vs December (all airports, 2023 on).

In [ ]:
rows = []
for label, sub in [("July", rec[rec.month == 7]), ("December", rec[rec.month == 12])]:
    s = sub[["n"] + BANDS].sum()
    for m, bs in OVER.items():
        rows.append({"month": label, "minutes": m, "share": sum(s[b] for b in bs) / s["n"]})
stair = pd.DataFrame(rows).pivot(index="minutes", columns="month", values="share")[["July", "December"]]
ax = stair.plot(marker="o", lw=3, color=[BLUE, ORANGE], logy=True,
                title="Chance of being more than X minutes late (log scale)")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.1%}")
plt.show()

prof = by(rec, ["month"])
print(f"Cancelled on the day: December {prof.cancelled[12]:.1%}, July {prof.cancelled[7]:.1%}")
g = grid.drop(index=[2020, 2021], level="year")
print("Five worst months for 3-hour+ delays:",
      " · ".join(f"{MONTHS[m - 1]} {y} {v:.1%}" for (y, m), v in g.late180.nlargest(5).items()))
stair

## Finding 4: when December goes wrong

The worst months for cancellations since the CAA started counting them (2018, Covid left out),
and how much one snowy weekend can do: flights between the UK and Munich in December 2023
(heavy snow closed Munich airport at the start of the month) vs the Decembers after.

In [ ]:
c18 = by(d[(d.year >= 2018) & ~d.year.between(2020, 2021)], ["year", "month"])
display(c18.nlargest(7, "cancelled")[["flights", "cancelled", "late15"]])

munich = by(decs[decs.dest == "MUNICH"], ["year"])
print("UK–Munich, December, cancelled:", " · ".join(f"{y} {v:.0%}" for y, v in munich.cancelled.items()))
inout = by(rec[rec.month == 12], ["ad"]).late15
print(f"December late, flights into the UK {inout['A']:.0%}, out of the UK {inout['D']:.0%}")

## Finding 5: the flip

European airports with lots of UK flights, July vs December (2023 on, 1,500+ flights in each
month). In July the beaches are the worst; in December it's the cold, busy hubs.

In [ ]:
mon = rec[(rec.region == "Europe") & rec.month.isin([7, 12])]
fl = by(mon, ["dest", "month"])[["late15", "flights"]].unstack()
fl.columns = [f"{a}_{'jul' if m == 7 else 'dec'}" for a, m in fl.columns]
fl = fl.dropna()
fl = fl[(fl.flights_jul >= 1500) & (fl.flights_dec >= 1500)]
fig, ax = plt.subplots(figsize=(6, 9))
for dest, r in fl.iterrows():
    top = r.late15_dec >= fl.late15_dec.nlargest(5).min()
    ax.plot([0, 1], [r.late15_jul, r.late15_dec], color=ORANGE if top else BLUE,
            alpha=1 if top else 0.35, lw=2.5 if top else 1)
    if top:
        ax.text(1.03, r.late15_dec, dest.title(), va="center")
ax.set(xticks=[0, 1], xticklabels=["July", "December"], xlim=(-0.1, 1.5),
       title="Flights late, July vs December, by European airport")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.show()
fl.sort_values("late15_dec", ascending=False)

## Finding 6: the Christmas trips

Destinations grouped by the kind of trip, December 2023–2025. Plus the airports that hardly exist
outside winter: the share of their UK flights (2023–2025) that were in December.

In [ ]:
TRIPS = {
    "Lapland": ["ROVANIEMI", "KITTILA", "IVALO", "ENONTEKIO", "KUUSAMO"],
    "Ski": ["GENEVA", "SALZBURG", "INNSBRUCK", "GRENOBLE", "CHAMBERY", "TURIN", "SION"],
    "Christmas markets": ["VIENNA", "PRAGUE", "BUDAPEST", "KRAKOW", "COLOGNE BONN", "NUREMBERG",
                          "BERLIN BRANDENBURG", "COPENHAGEN", "STRASBOURG", "TALLIN"],
    "Family hubs": ["DUBLIN", "AMSTERDAM", "PARIS (CHARLES DE GAULLE)", "FRANKFURT MAIN", "MUNICH"],
    "Canaries": ["TENERIFE (SURREINA SOFIA)", "ARRECIFE", "LAS PALMAS", "FUERTEVENTURA"],
    "Spain & Portugal sun": ["MALAGA", "ALICANTE", "FARO", "FUNCHAL", "PALMA DE MALLORCA"],
}
trips = pd.DataFrame({t: summarise(decs[decs.dest.isin(v)]) for t, v in TRIPS.items()}).T
hbar(trips.late15.sort_values(ascending=False), "Flights late, December 2023–2025, by kind of trip",
     hot="Lapland", labels=[f"{v:.0%} · 1 in {1 / c:.0f} cancelled"
                            for v, c in trips.sort_values("late15", ascending=False)[["late15", "cancelled"]].values])

full = d[d.ym.between(202301, 202512) & (d.region != "UK")]
season = (full[full.month == 12].groupby("dest").n.sum() / full.groupby("dest").n.sum()).dropna()
season = season[full[full.month == 12].groupby("dest").n.sum() >= 300]
hbar(season.nlargest(8).rename(str.title), "Share of each airport's UK flights that were in December, 2023–2025")
trips

## Finding 7: the airports

Every UK airport with 3,000+ flights in the last three Decembers: late, three hours late and
cancelled. Then who does the cancelling at the airports at the top.

In [ ]:
apd = by(decs, ["airport"])
apd = apd[apd.flights >= 3000]
apd["july_late15"] = by(rec[rec.month == 7], ["airport"]).late15
apd["one_in_cancelled"] = (1 / apd.cancelled).round()
hbar(apd.cancelled.sort_values(ascending=False).rename(str.title), "December flights cancelled, by UK airport",
     fmt="{:.1%}",
     labels=[f"1 in {1 / v:.0f}" for v in apd.cancelled.sort_values(ascending=False)])
top4 = apd.cancelled.nlargest(4).index
who = (decs[decs.airport.isin(top4)].groupby(["airport", "brand"])[["n", "cancelled"]].sum()
       .sort_values("cancelled", ascending=False).groupby(level="airport").head(3))
display(who)
apd.sort_values("late15", ascending=False)

## Finding 8: the routes

Airport + destination, both directions, December 2023–2025, 900+ flights. And flying home
inside the UK: domestic routes with the most cancellations (small numbers: read as rough).

In [ ]:
rt = by(decs[decs.region != "UK"], ["airport", "dest"])
rt = rt[rt.flights >= 900]
print("Latest:"); display(rt.nlargest(5, "late15"))
print("Most punctual:"); display(rt.nsmallest(5, "late15"))
mc = rt.nlargest(10, "cancelled")
print(f"Of the 10 most-cancelled routes, {(mc.index.get_level_values('dest') == 'AMSTERDAM').sum()} go to Amsterdam")
display(mc)

split = by(decs, ["region"])
print(f"December, 3 h+: long-haul 1 in {1 / split.late180['Long-haul']:.0f}, "
      f"Europe 1 in {1 / split.late180['Europe']:.0f}; domestic cancelled 1 in {1 / split.cancelled['UK']:.0f}")
dom = by(decs[decs.region == "UK"], ["airport", "dest"])
dom[dom.flights >= 150].nlargest(10, "cancelled")

## Finding 9: the airlines

The whole last year vs December for the big airlines, then the December trade-off: some
airlines cancel, others fly late. Aer Lingus is left out of the cancellation comparison: its
regional flights show more cancellations than flights at Belfast City in the CAA files.

In [ ]:
al = by(l12, ["brand"])
al_dec = by(decs, ["brand"])
big = al[al.flights >= 20000].index.intersection(al_dec.index)
t = pd.DataFrame({"Last 12 months": al.loc[big, "late15"], "December": al_dec.loc[big, "late15"]})
t = t.sort_values("December")
ax = t.plot.barh(color=[BLUE, ORANGE], figsize=(10, 7), title="Flights late: the last year vs December")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.show()

trade = al_dec.loc[["Loganair", "KLM", "Lufthansa", "British Airways", "easyJet", "TUI", "Ryanair", "Jet2"]]
trade = trade.assign(one_in_3h=(1 / trade.late180).round(), one_in_cancelled=(1 / trade.cancelled).round())
display(trade[["flights", "late15", "one_in_3h", "one_in_cancelled"]])

sc = by(decs, ["sc"])
print(f"December: charter flights {sc.late15['C']:.0%} late, scheduled {sc.late15['S']:.0%}")

## If it happens to you: last December in numbers

December 2025, all UK airports. The rules (UK261 / EU261): arrive 3 hours or more late and you
can be owed £220 / £350 / £520 (€250 / €400 / €600) depending on distance, unless the cause was
extraordinary (weather, air traffic control strikes, security). Food, drink, a hotel if needed,
and a refund or a new flight on a cancellation are owed either way. Check the
[CAA's page](https://www.caa.co.uk/passengers-and-public/resolving-travel-problems/delays-and-cancellations/delays/)
for the details.

In [ ]:
last = summarise(d[d.ym == 202512])
print(f"December 2025: {last.flights:,.0f} take-offs and landings, "
      f"{last.late180 * last.flights:,.0f} more than 3 hours late, "
      f"{last.cancelled / (1 - last.cancelled) * last.flights:,.0f} cancelled")

## Summary

| | |
|---|---|
| December 2010 | 50% of flights late, 1 in 27 more than 3 hours late |
| December the worst month of its year | 2006, and every year from 2009 to 2014 |
| Since 2023 (same 8 airports) | July worst (43%), December 5th (33%) |
| Decembers 2022 → 2025 | 47% → 39% → 34% → 27% late |
| 3 hours+, July vs December (2023 on) | 0.9% both |
| Cancelled, December vs July | 1.7% vs 1.2% |
| Worst December airports in Europe | Helsinki 46%, Munich 42%, Lisbon 40% |
| Best | Malaga 27%, Alicante 25%, Faro 24% |
| Lapland in December | 48% late, 1 in 254 cancelled |
| Most-cancelled routes | 7 of the top 10 go to Amsterdam |
| Cancel or fly late (December) | Lufthansa 1 in 22 cancelled, Ryanair 1 in 245 |

December isn't the worst month to fly any more: summer is. But December has just as many
3-hour delays as July, more cancellations, and four of the five worst months for long delays
in twenty years. Where you fly matters as much as when.

**Limits:** UK airports only (a flight from Munich to Vienna isn't in here). Monthly data, so no
Christmas week. How late a flight was, not why. No cancellations before 2018. Small routes
(Lapland, Chambéry, the Scottish islands) are a few hundred flights a year: read them as rough.